<a href="https://colab.research.google.com/github/logankim0913/EE_467_Final_Project/blob/standardScaler/phase1_standardscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import os

file_path = os.path.join("DictionaryBruteForce.pcap.csv")
#read the csv files and print the first 10 rows of each
bruteforcedata = pd.read_csv(file_path)
bruteforcedata.head(10)

spoofdata = pd.read_csv("DNS_Spoofing.pcap.csv")
spoofdata.head(10)

benigndata = pd.read_csv("BenignTraffic.pcap.csv")
benigndata.head(10)

# #Preview statistics of the data
print("Brute Force Data Statistics:")
print(bruteforcedata.describe())
print("\nDNS Spoofing Data Statistics:")
print(spoofdata.describe())
print("\nBenign Traffic Data Statistics:")
print(benigndata.describe())

Brute Force Data Statistics:
       Header_Length  Protocol Type  Time_To_Live           Rate  \
count   13064.000000   13064.000000  13064.000000   13064.000000   
mean       23.883762       7.632961     93.120070    1732.280240   
std         7.323148       4.064012     34.447147   24243.212110   
min         0.800000       0.000000     15.600000       0.000040   
25%        18.800000       6.000000     64.000000      52.411901   
50%        24.800000       6.000000     83.100000     106.396493   
75%        30.400000       6.000000    112.500000     249.954800   
max        44.800000      17.000000    247.000000  762600.727273   

       fin_flag_number  syn_flag_number  rst_flag_number  psh_flag_number  \
count     13064.000000     13064.000000     13064.000000     13064.000000   
mean          0.026607         0.046249         0.010257         0.251944   
std           0.075118         0.089139         0.036774         0.195413   
min           0.000000         0.000000         0.

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# handle infinite values before scaling
# drop rows with infinity
benigndata_cleaned = benigndata.replace([np.inf, -np.inf], np.nan).dropna()
spoofdata_cleaned = spoofdata.replace([np.inf, -np.inf], np.nan).dropna()

#obtaining features as a list for each chosen class
bfd_scaled = bruteforcedata.copy()
categorical_col_bfd = []
for col in bfd_scaled.columns:
  categorical_col_bfd.append(col)
print(categorical_col_bfd)
benign_scaled = benigndata_cleaned.copy() #no inf
categorical_col_benign = []
for col in benign_scaled.columns:
  categorical_col_benign.append(col)

spoof_scaled = spoofdata_cleaned.copy() #no inf
categorical_col_spoof = []
for col in spoof_scaled.columns:
  categorical_col_spoof.append(col)

#Standard scaling (no outlier removal yet)
bfd_feat_std = pd.DataFrame(StandardScaler().fit_transform(bruteforcedata[categorical_col_bfd]), columns=categorical_col_bfd)
benign_feat_std = pd.DataFrame(StandardScaler().fit_transform(benigndata_cleaned[categorical_col_benign]), columns=categorical_col_benign)
spoof_feat_std = pd.DataFrame(StandardScaler().fit_transform(spoofdata_cleaned[categorical_col_spoof]), columns=categorical_col_spoof)

# Outlier removal logic
remove_outliers = True
outlier_threshold = 5

#Dictionary Brute Force Data
if remove_outliers:
    outlier_indices_bfd = set()
    for col_name in bfd_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = bfd_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indices with `outlier_indices_bfd`
        outlier_indices_bfd.update(out_indices.tolist())
    #Remove outliers
    bfd_feat_std = bfd_feat_std.drop(index=list(outlier_indices_bfd))

    # Benign Data
    outlier_indices_benign = set()
    for col_name in benign_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = benign_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        # Merge these indices with `outlier_indices_benign`
        outlier_indices_benign.update(out_indices.tolist())
    #Remove outliers
    benign_feat_std = benign_feat_std.drop(index=list(outlier_indices_benign))

    # DNS Spoofing Data
    outlier_indices_spoof = set()
    for col_name in spoof_feat_std.columns:
        #Obtain column data as 2D NumPy array
        col_data = spoof_feat_std[col_name].values.astype(float)
        #Get indices of records with outlier column values
        out_indices = np.where(np.abs(col_data) > outlier_threshold)[0]
        #Merge these indives with 'outlier_indices_spoof"
        outlier_indices_spoof.update(out_indices.tolist())
    #Remove outliers
    spoof_feat_std = spoof_feat_std.drop(index=list(outlier_indices_spoof))

print("Brute Force Scaled Data (without outliers):")
print(bfd_feat_std.head())
print("\nBenign Scaled Data (without outliers):")
print(benign_feat_std.head())
print("\nSpoof Scaled Data (without outliers):")
print(spoof_feat_std.head())

['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance']
Brute Force Scaled Data (without outliers):
   Header_Length  Protocol Type  Time_To_Live      Rate  fin_flag_number  \
0       0.452861      -0.401825      0.701970 -0.070212         2.308350   
1       0.835224      -0.401825      0.150379 -0.070211        -0.354221   
2      -0.584984      -0.401825     -0.232831 -0.066449        -0.354221   
3       0.288990      -0.401825      0.710679 -0.070138        -0.354221   
4      -0.967348       2.304963      1.070665 -0.067000        -0.354221   

   syn_flag_number  rst_flag_number  psh_flag_number  ack_fl

In [6]:
#NOT CURRENTLY APPLIED TO PHASE 1
#Just a start to potential Robust Scaling if eventually needed
from sklearn.preprocessing import StandardScaler, RobustScaler

# Copy original dataset
bruteforce_scaled = bruteforcedata.copy()

# Convert transaction time from seconds to hour-of-day (0-23) then scale with `StandardScaler`
time_in_hours = bruteforcedata["Time_To_Live"] / (60 * 60)
hour_of_day = time_in_hours % 24
time_scaled = StandardScaler().fit_transform(hour_of_day.values.reshape(-1, 1))
bruteforce_scaled["Time_To_Live"] = time_scaled

# Rescale and override transaction amount column with `RobustScaler`
amount_scaled = RobustScaler().fit_transform(bruteforcedata["Tot size"].values.reshape(-1, 1))
bruteforce_scaled["Tot size"] = amount_scaled

# Review scaling results
print("Scaled Brute Force Data:")
bruteforce_scaled[["Tot size", "Time_To_Live"]].head(5)

Scaled Brute Force Data:


,Tot size,Time_To_Live
0,-0.262547,0.701970
1,-0.270037,0.150379
2,-0.151685,-0.232831
3,-0.045318,0.710679
4,2.498502,1.070665


In [ ]:
#Test split
